<h1>Import dataset and modules<h1>

In [2]:
import numpy, pandas, matplotlib, sklearn, seaborn, ipykernel

import pandas as pd

df = pd.read_csv("diabetic_data.csv", na_values="?")

/var/folders/wy/ymq6dx6n0z76tgh43whw2dxr0000gn/T/ipykernel_61365/3955573027.py:5: DtypeWarning: Columns (0: payer_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("diabetic_data.csv", na_values="?")


<h2>Grouping the diag_1 variable and dropping diag_2, diag_3 and number_diagnoses<h2>

As the diag_1 shows the primary diagnosis accredited for the stay of the patient, we assume this diagnosis is placed during the admission of the patient to the hospital. In this way this would be data that would be avaivable to the scheduling office at the hospital when the patient is addmited.

Doing this also helps simplify the model, so it does not overfit.

In [3]:
# Defines the buckets for the diag_1 values
d1_bucket_def = {
    1: "Infectious and Parasitic Diseases",
    140: "Neoplasms",
    240: "Endocrine, Nutritional and Metabolic Diseases, and Immunity Disorders",
    280: "Diseases of the Blood and Blood-forming Organs",
    290: "Mental Disorders",
    320: "Diseases of the Nervous System and Sense Organs",
    390: "Diseases of the Circulatory System",
    460: "Diseases of the Respiratory System",
    520: "Diseases of the Digestive System",
    580: "Diseases of the Genitourinary System",
    630: "Complications of Pregnancy, Childbirth, and the Puerperium",
    680: "Diseases of the Skin and Subcutaneous Tissue",
    710: "Diseases of the Musculoskeletal System and Connective Tissue",
    740: "Congenital Anomalies",
    760: "Certain Conditions Originating in the Perinatal Period",
    780: "Symptoms, Signs, and Ill-defined Conditions",
    800: "Injury and Poisoning",
}

# Extracts and converts the variable data into a string, drops white spaces and upparcases all letters
codes = df["diag_1"].astype("string").str.strip().str.upper()

# Extract the numeric value before the dot using a regex. An "8" is seen as "008". Requires digits as an input and gives missing values if it gets a letters or symbols a part form "."
numeric_category = pd.to_numeric(
    codes.str.extract(r"^([0-9]{1,3})(?:\.[0-9]+)?$", expand=False),
    errors="coerce",
)

# assigns a new column to df, by comparing the extracted number to the buckets, and then outputing the name of the bucket
df["diag_1_category"] = pd.cut(
    numeric_category,
    bins=[*d1_bucket_def, 1000],
    labels=list(d1_bucket_def.values()),
    right=False,
).astype("string")

# Assign the supplementary codes by prefix.
df.loc[codes.str.startswith("E", na=False), "diag_1_category"] = (
    "External Causes of Injury and Poisoning"
)
df.loc[codes.str.startswith("V", na=False), "diag_1_category"] = (
    "Factors Influencing Health Status and Contact with Health Services"
)

print(df["diag_1_category"].value_counts(dropna=False))

#overwrites the old diag_1 values
df['diag_1'] = df["diag_1_category"]


diag_1_category
Diseases of the Circulatory System                                       30336
Endocrine, Nutritional and Metabolic Diseases, and Immunity Disorders    11459
Diseases of the Respiratory System                                       10407
Diseases of the Digestive System                                          9208
Symptoms, Signs, and Ill-defined Conditions                               7636
Injury and Poisoning                                                      6974
Diseases of the Genitourinary System                                      5078
Diseases of the Musculoskeletal System and Connective Tissue              4957
Neoplasms                                                                 3433
Infectious and Parasitic Diseases                                         2768
Diseases of the Skin and Subcutaneous Tissue                              2530
Mental Disorders                                                          2262
Factors Influencing Health Status an

The variable now contains 19 distinct values one of which is NA, the others are descriptors of categories of diagnosies.

We drop diag_2, diag_3 and number_diagnoses because we cannot assume at what point these diagnosis were placed, thus this data might not be avaibale at the time of the model's real world use.

In [4]:
df = df.drop(columns = ['diag_2','diag_3','number_diagnoses'])

Rename the diag_1 column

In [5]:
df = df.rename(columns={'diag_1' : 'diag_cat'})